In [1]:

import pandas as pd
from pathlib import Path

RAW_PATH = Path("../data/silver/transactions_raw.parquet")
df = pd.read_parquet(RAW_PATH)

df.shape


(1048575, 11)

In [2]:

print("Valeurs manquantes par colonne:")
print(df.isnull().sum())

print("\nNombre de doublons:", df.duplicated().sum())


Valeurs manquantes par colonne:
step              0
type              0
amount            0
nameOrig          0
oldbalanceOrg     0
newbalanceOrig    0
nameDest          0
oldbalanceDest    0
newbalanceDest    0
isFraud           0
isFlaggedFraud    0
dtype: int64

Nombre de doublons: 0


In [3]:

invalid_amount = (df["amount"] <= 0).sum()
print("Transactions avec amount <= 0 :", invalid_amount)


Transactions avec amount <= 0 : 0


In [4]:

df["balance_diff_orig"] = df["oldbalanceOrg"] - df["amount"] - df["newbalanceOrig"]
df["balance_diff_dest"] = df["oldbalanceDest"] + df["amount"] - df["newbalanceDest"]

incoh_orig = (df["balance_diff_orig"].abs() > 1e-6).sum()
incoh_dest = (df["balance_diff_dest"].abs() > 1e-6).sum()

print("Incohérences balance origine:", incoh_orig)
print("Incohérences balance destination:", incoh_dest)


Incohérences balance origine: 833596
Incohérences balance destination: 801560


In [6]:

df_clean = df.drop_duplicates()
df_clean = df_clean[df_clean["amount"] > 0]

CLEAN_PATH = Path("../data/silver/transactions_clean.parquet")
df_clean.to_parquet(CLEAN_PATH, index=False)

print(" Sauvegardé:", CLEAN_PATH)
print("Avant:", df.shape, "| Après:", df_clean.shape)


 Sauvegardé: ..\data\silver\transactions_clean.parquet
Avant: (1048575, 13) | Après: (1048575, 13)


In [7]:
import json
from pathlib import Path

report = {
    "rows": int(df.shape[0]),
    "cols": int(df.shape[1]),
    "missing_values_total": int(df.isnull().sum().sum()),
    "duplicates": int(df.duplicated().sum()),
    "invalid_amount_leq_0": int((df["amount"] <= 0).sum()),
    "incoh_orig": int((df["balance_diff_orig"].abs() > 1e-6).sum()),
    "incoh_dest": int((df["balance_diff_dest"].abs() > 1e-6).sum()),
}

Path("../reports").mkdir(exist_ok=True)

with open("../reports/data_quality_report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

print("✅ Report sauvegardé dans reports/data_quality_report.json")
report

✅ Report sauvegardé dans reports/data_quality_report.json


{'rows': 1048575,
 'cols': 13,
 'missing_values_total': 0,
 'duplicates': 0,
 'invalid_amount_leq_0': 0,
 'incoh_orig': 833596,
 'incoh_dest': 801560}